In [1]:

# ============================================================
# DAY 6 — PIT-STOP SCENARIO SIMULATOR
# Setup and data loading
# ============================================================

from pathlib import Path
import itertools
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ------------------------------------------------------------
# 1. Project paths
# ------------------------------------------------------------

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"

print("Project root:", ROOT)
print("Processed data folder:", PROCESSED)

# ------------------------------------------------------------
# 2. Load datasets
# ------------------------------------------------------------

pace = pd.read_csv(
    PROCESSED / "pace_clean_candidate_2023_2024.csv"
)

pit_stops = pd.read_csv(
    PROCESSED / "pit_stops_all_2023_2024.csv"
)

# ------------------------------------------------------------
# 3. Basic validation
# ------------------------------------------------------------

required_pace_columns = {
    "season", "round", "lap", "stint_lap",
    "reference_pace", "delta"
}

required_pit_columns = {
    "season", "round", "driver_id", "lap",
    "stop", "duration_seconds"
}

assert required_pace_columns.issubset(pace.columns), (
    "Missing required columns in pace dataset"
)

assert required_pit_columns.issubset(pit_stops.columns), (
    "Missing required columns in pit-stop dataset"
)

# ------------------------------------------------------------
# 4. Create race identifier
# ------------------------------------------------------------

pace["race_id"] = (
    pace["season"].astype(str)
    + "_R"
    + pace["round"].astype(str)
)

pit_stops["race_id"] = (
    pit_stops["season"].astype(str)
    + "_R"
    + pit_stops["round"].astype(str)
)

# ------------------------------------------------------------
# 5. Apply the same 200-row race eligibility rule
# ------------------------------------------------------------

race_counts = pace.groupby("race_id").size()

eligible_races = race_counts[
    race_counts >= 200
].index

data = pace[
    pace["race_id"].isin(eligible_races)
].copy()

data = data.reset_index(drop=True)

# ------------------------------------------------------------
# 6. Prepare linear and quadratic features
# ------------------------------------------------------------

X_linear = data[["lap", "stint_lap"]].copy()

X_quadratic = X_linear.copy()

X_quadratic["stint_lap_sq"] = (
    X_quadratic["stint_lap"] ** 2
)

y = data["delta"].to_numpy()
groups = data["race_id"].to_numpy()

# ------------------------------------------------------------
# 7. Fit selected quadratic Ridge model
# ------------------------------------------------------------

final_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

final_model.fit(X_quadratic, y)

# ------------------------------------------------------------
# 8. Display setup summary
# ------------------------------------------------------------

print("\nSETUP COMPLETE")
print("------------------------------")
print("Pace dataset:", pace.shape)
print("Pit-stop dataset:", pit_stops.shape)
print("Eligible races:", data["race_id"].nunique())
print("Eligible modelling rows:", len(data))
print("Linear features:", X_linear.columns.tolist())
print("Quadratic features:", X_quadratic.columns.tolist())

print("\nModel fitted:", final_model)


Project root: d:\projects\f1-project
Processed data folder: d:\projects\f1-project\data\processed

SETUP COMPLETE
------------------------------
Pace dataset: (30054, 14)
Pit-stop dataset: (1737, 9)
Eligible races: 35
Eligible modelling rows: 29901
Linear features: ['lap', 'stint_lap']
Quadratic features: ['lap', 'stint_lap', 'stint_lap_sq']

Model fitted: Pipeline(steps=[('standardscaler', StandardScaler()), ('ridge', Ridge())])


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneGroupOut

ROOT = Path(r"D:\projects\f1-project")
PROCESSED = ROOT / "data" / "processed"

print("FILES IN data/processed")
for path in sorted(PROCESSED.glob("*.csv")):
    columns = pd.read_csv(path, nrows=0).columns.tolist()
    with open(path, encoding="utf-8") as f:
        n_rows = sum(1 for _ in f) - 1
    print(f"{path.name:45s} rows={n_rows:>7,}  columns={columns}")

pred_path = PROCESSED / "pred_rows_2023_2024.csv"
if not pred_path.exists():
    print("\npred_rows_2023_2024.csv not found. Send me the list above and tell me which file is your modelling file.")
else:
    d = pd.read_csv(pred_path)
    d = d[d["keep"]].reset_index(drop=True)
    d["stint_lap_sq"] = d["stint_lap"] ** 2
    y = d["delta"].values
    groups = d["race"].values
    print(f"\nMODEL COMPARISON: {len(d):,} rows, {d['race'].nunique()} races, leave-one-race-out")

    feature_sets = {
        "train mean (baseline)": [],
        "lap + stint_lap": ["lap", "stint_lap"],
        "lap + stint_lap + stint_lap^2": ["lap", "stint_lap", "stint_lap_sq"],
    }
    per_race, rows = {}, []
    for name, feats in feature_sets.items():
        errors = np.zeros(len(d))
        for train_idx, test_idx in LeaveOneGroupOut().split(d, y, groups):
            if feats:
                model = LinearRegression().fit(d.loc[train_idx, feats], y[train_idx])
                predicted = model.predict(d.loc[test_idx, feats])
            else:
                predicted = np.full(len(test_idx), y[train_idx].mean())
            errors[test_idx] = y[test_idx] - predicted
        per_race[name] = pd.Series(np.abs(errors)).groupby(groups).mean()
        rows.append({"model": name, "MAE": np.abs(errors).mean(),
                     "RMSE": np.sqrt((errors ** 2).mean()),
                     "median_abs_err": np.median(np.abs(errors))})

    table = pd.DataFrame(rows).set_index("model")
    base = per_race["train mean (baseline)"]
    table["races_beating_baseline"] = [int((per_race[m] < base).sum()) if m != "train mean (baseline)" else np.nan
                                       for m in table.index]
    print(table.round(3))

    print("\nCOEFFICIENTS (fit on all rows; seconds per lap)")
    for feats in (["lap", "stint_lap"], ["lap", "stint_lap", "stint_lap_sq"]):
        m = LinearRegression().fit(d[feats], d["delta"])
        print(feats, {f: round(float(c), 5) for f, c in zip(feats, m.coef_)}, "| intercept", round(float(m.intercept_), 3))
    print("\nstint_lap: max", int(d["stint_lap"].max()),
          "| share of rows with stint_lap > 30:", round(float((d["stint_lap"] > 30).mean()), 3))

FILES IN data/processed
fetch_qc_2023_2024.csv                        rows=     36  columns=['season', 'round', 'rows', 'api_total', 'duplicates', 'laps', 'drivers']
laps_2023_2024_first5.csv                     rows= 10,050  columns=['season', 'round', 'lap', 'driver_id', 'position', 'lap_time']
laps_all_2023_2024.csv                        rows= 50,960  columns=['season', 'round', 'lap', 'driver_id', 'position', 'lap_time']
laps_features_2023_2024_first5.csv            rows= 10,050  columns=['season', 'round', 'lap', 'driver_id', 'position', 'lap_time', 'is_pit_lap', 'lap_time_seconds', 'race_median', 'pace_ratio', 'odd_pace', 'ref_prelim', 'delta_prelim', 'near_pit', 'is_clean', 'ref_pace', 'delta', 'n_clean', 'usable_ref', 'last_stop_lap', 'stint_lap']
laps_with_pit_flags_2023_2024_first5.csv      rows= 10,050  columns=['season', 'round', 'lap', 'driver_id', 'position', 'lap_time', 'is_pit_lap']
model_comparison_2023_2024.csv                rows=      8  columns=['Dataset', 'Model'

In [3]:
import itertools
import numpy as np
import pandas as pd


def stint_lap_series(n_laps, pit_laps):
    """stint_lap for laps 1..n_laps. A stop on lap P belongs to the stint it ends:
    lap P is the last lap of the old stint, lap P+1 is lap 1 of the new stint."""
    stint = np.zeros(n_laps, dtype=int)
    count = 0
    pits = set(pit_laps)
    for lap in range(1, n_laps + 1):
        count += 1
        stint[lap - 1] = count
        if lap in pits:
            count = 0
    return stint


def simulate(n_laps, pit_laps, reference_pace, fuel_per_lap, wear_per_lap,
             wear_sq_per_lap, pit_loss, intercept=0.0):
    """Estimated race time (seconds) for one driver under one pit strategy.
    lap time = reference + intercept + fuel*lap + wear*stint_lap + wear_sq*stint_lap^2
    plus pit_loss seconds for every stop."""
    laps = np.arange(1, n_laps + 1)
    stint = stint_lap_series(n_laps, pit_laps)
    lap_times = (reference_pace + intercept + fuel_per_lap * laps
                 + wear_per_lap * stint + wear_sq_per_lap * stint ** 2)
    return float(lap_times.sum() + pit_loss * len(pit_laps))


def candidate_strategies(n_laps, n_stops, min_stint=8):
    """Every set of pit laps with at least min_stint laps between stops."""
    for pits in itertools.combinations(range(min_stint, n_laps - min_stint + 1), n_stops):
        if np.diff([0, *pits, n_laps]).min() >= min_stint:
            yield pits


def best_by_stops(n_laps, params, max_stops=2, min_stint=8):
    out = []
    for k in range(max_stops + 1):
        time_s, pits = min((simulate(n_laps, p, **params), p)
                           for p in candidate_strategies(n_laps, k, min_stint))
        out.append({"stops": k, "best_pit_laps": pits, "race_time_s": round(time_s, 1)})
    return pd.DataFrame(out)


# ---------- tests ----------
assert list(stint_lap_series(6, [3])) == [1, 2, 3, 1, 2, 3]
flat = dict(reference_pace=90.0, fuel_per_lap=0.0, wear_per_lap=0.0, wear_sq_per_lap=0.0, pit_loss=0.0)
assert simulate(50, [], **flat) == 50 * 90.0
assert simulate(50, [25], **{**flat, "pit_loss": 22.0}) == 50 * 90.0 + 22.0
print("tests passed")

# ---------- toy demonstration: NOT from the data ----------
N_LAPS = 55
toy = dict(reference_pace=90.0, fuel_per_lap=-0.06, wear_per_lap=0.05, wear_sq_per_lap=0.0, pit_loss=22.0)
print("\nBest strategy by number of stops (0 stops is illegal in a dry race; shown for reference only)")
print(best_by_stops(N_LAPS, toy))

print("\nGain of the best 2-stop over the best 1-stop, in seconds (positive = 2-stop faster)")
grid = {}
for wear in (0.02, 0.05, 0.10):
    for loss in (18, 22, 26, 30):
        params = {**toy, "wear_per_lap": wear, "pit_loss": float(loss)}
        best = best_by_stops(N_LAPS, params).set_index("stops")["race_time_s"]
        grid[(f"wear {wear} s/lap", f"pit loss {loss} s")] = round(best[1] - best[2], 1)
print(pd.Series(grid).unstack())

tests passed

Best strategy by number of stops (0 stops is illegal in a dry race; shown for reference only)
   stops best_pit_laps  race_time_s
0      0            ()       4934.6
1      1         (27,)       4918.8
2      2      (18, 36)       4928.2

Gain of the best 2-stop over the best 1-stop, in seconds (positive = 2-stop faster)
                 pit loss 18 s  pit loss 22 s  pit loss 26 s  pit loss 30 s
wear 0.02 s/lap          -12.9          -16.9          -20.9          -24.9
wear 0.05 s/lap           -5.4           -9.4          -13.4          -17.4
wear 0.1 s/lap             7.2            3.2           -0.8           -4.8


In [4]:

# Inspect predicted delta across realistic race and stint laps

race_laps = np.arange(
    int(data["lap"].min()),
    int(data["lap"].max()) + 1
)

stint_lengths = [5, 10, 15, 20, 25, 30, 35, 40]

prediction_rows = []

for stint in stint_lengths:
    check = pd.DataFrame({
        "lap": race_laps,
        "stint_lap": stint
    })

    check["stint_lap_sq"] = check["stint_lap"] ** 2

    check["predicted_delta"] = final_model.predict(
        check[["lap", "stint_lap", "stint_lap_sq"]]
    )

    prediction_rows.append({
        "stint_lap": stint,
        "min_predicted_delta": check["predicted_delta"].min(),
        "max_predicted_delta": check["predicted_delta"].max(),
        "mean_predicted_delta": check["predicted_delta"].mean()
    })

prediction_check = pd.DataFrame(prediction_rows)

print("OBSERVED FEATURE RANGES")
print("Race lap:", data["lap"].min(), "to", data["lap"].max())
print("Stint lap:", data["stint_lap"].min(), "to", data["stint_lap"].max())

print("\nPREDICTED DELTA BY STINT LENGTH")
display(prediction_check.round(3))


OBSERVED FEATURE RANGES
Race lap: 7 to 72
Stint lap: 2 to 58

PREDICTED DELTA BY STINT LENGTH


,stint_lap,min_predicted_delta,max_predicted_delta,mean_predicted_delta
0,5,-3.039,-0.744,-1.892
1,10,-2.761,-0.466,-1.613
2,15,-2.556,-0.261,-1.408
3,20,-2.423,-0.128,-1.276
4,25,-2.364,-0.069,-1.217
5,30,-2.378,-0.083,-1.230
6,35,-2.464,-0.170,-1.317
7,40,-2.624,-0.329,-1.477


In [5]:

# ============================================================
# CELL D — LINEAR VS QUADRATIC MODEL DIAGNOSTICS
# ============================================================

# Fit linear Ridge on the same 35-race dataset
linear_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

linear_model.fit(X_linear, y)

def get_original_coefficients(model):
    scaler = model.named_steps["standardscaler"]
    ridge = model.named_steps["ridge"]

    coef = ridge.coef_ / scaler.scale_

    intercept = ridge.intercept_ - np.sum(
        ridge.coef_ * scaler.mean_ / scaler.scale_
    )

    return intercept, coef

linear_intercept, linear_coef = get_original_coefficients(
    linear_model
)

quadratic_intercept, quadratic_coef = get_original_coefficients(
    final_model
)

print("LINEAR RIDGE")
print("Intercept:", round(linear_intercept, 6))
print("Lap coefficient:", round(linear_coef[0], 6))
print("Stint-lap coefficient:", round(linear_coef[1], 6))

print("\nQUADRATIC RIDGE")
print("Intercept:", round(quadratic_intercept, 6))
print("Lap coefficient:", round(quadratic_coef[0], 6))
print("Stint-lap coefficient:", round(quadratic_coef[1], 6))
print("Squared coefficient:", round(quadratic_coef[2], 6))

# Inspect the observed stint-lap distribution
print("\nSTINT-LAP DISTRIBUTION")
display(
    data["stint_lap"].describe().to_frame("stint_lap").round(2)
)

for threshold in [15, 20, 25, 30, 35, 40]:
    share = (data["stint_lap"] > threshold).mean() * 100
    print(
        f"Rows with stint_lap > {threshold}: "
        f"{share:.2f}%"
    )


LINEAR RIDGE
Intercept: -0.394763
Lap coefficient: -0.035437
Stint-lap coefficient: 0.017258

QUADRATIC RIDGE
Intercept: -0.848157
Lap coefficient: -0.035306
Stint-lap coefficient: 0.077536
Squared coefficient: -0.00146

STINT-LAP DISTRIBUTION


,stint_lap
count,29901.00
mean,15.88
std,9.80
min,2.00
25%,8.00
50%,14.00
75%,21.00
max,58.00


Rows with stint_lap > 15: 43.79%
Rows with stint_lap > 20: 27.27%
Rows with stint_lap > 25: 16.03%
Rows with stint_lap > 30: 8.91%
Rows with stint_lap > 35: 4.68%
Rows with stint_lap > 40: 2.23%


In [6]:

# ============================================================
# CELL E — FIND SUITABLE HISTORICAL RACES
# ============================================================

# Driver-race coverage in our modelling dataset
driver_coverage = (
    data.groupby(["season", "round", "race_id", "driver_id"])
    .agg(
        clean_rows=("delta", "size"),
        first_lap=("lap", "min"),
        last_lap=("lap", "max"),
        reference_pace=("reference_pace", "first")
    )
    .reset_index()
)

# Actual pit-stop coverage
pit_coverage = (
    pit_stops.groupby(["season", "round", "race_id", "driver_id"])
    .agg(
        actual_stops=("lap", "count"),
        actual_pit_laps=("lap", lambda x: sorted(x.tolist()))
    )
    .reset_index()
)

# Join both datasets
race_candidates = driver_coverage.merge(
    pit_coverage,
    on=["season", "round", "race_id", "driver_id"],
    how="inner"
)

# Prefer drivers with sufficient clean laps and at least one actual stop
race_candidates = race_candidates[
    (race_candidates["clean_rows"] >= 20)
    & (race_candidates["actual_stops"] >= 1)
].copy()

race_candidates = race_candidates.sort_values(
    ["season", "round", "clean_rows"],
    ascending=[True, True, False]
)

print("Suitable driver-race combinations:", len(race_candidates))

display(
    race_candidates[
        [
            "season",
            "round",
            "driver_id",
            "clean_rows",
            "first_lap",
            "last_lap",
            "reference_pace",
            "actual_stops",
            "actual_pit_laps"
        ]
    ].head(15)
)


Suitable driver-race combinations: 648


,season,round,driver_id,clean_rows,first_lap,last_lap,reference_pace,actual_stops,actual_pit_laps
1,2023,1,alonso,46,7,57,99.5552,2,"[14, 34]"
5,2023,1,hamilton,46,7,57,99.3350,2,"[12, 30]"
9,2023,1,max_verstappen,46,7,57,97.9954,2,"[14, 36]"
12,2023,1,perez,46,7,57,98.7144,2,"[17, 34]"
13,2023,1,russell,46,7,57,99.3972,2,"[13, 31]"
14,2023,1,sainz,46,7,57,99.1124,2,"[13, 31]"
16,2023,1,stroll,46,7,57,99.6240,2,"[15, 30]"
0,2023,1,albon,45,7,57,100.3368,3,"[11, 26, 40]"
2,2023,1,bottas,45,7,57,100.0948,2,"[11, 29]"
4,2023,1,gasly,45,7,57,100.7806,3,"[9, 25, 40]"


In [7]:

# ============================================================
# CELL F — INSPECT 2023 BAHRAIN GP, VERSTAPPEN
# ============================================================

all_laps = pd.read_csv(
    PROCESSED / "laps_all_2023_2024.csv"
)

# Selected race and driver
SEASON = 2023
ROUND = 1
DRIVER = "max_verstappen"

raw_driver = all_laps[
    (all_laps["season"] == SEASON)
    & (all_laps["round"] == ROUND)
    & (all_laps["driver_id"] == DRIVER)
].copy()

model_driver = data[
    (data["season"] == SEASON)
    & (data["round"] == ROUND)
    & (data["driver_id"] == DRIVER)
].copy()

pit_driver = pit_stops[
    (pit_stops["season"] == SEASON)
    & (pit_stops["round"] == ROUND)
    & (pit_stops["driver_id"] == DRIVER)
].copy()

raw_driver = raw_driver.sort_values("lap")
model_driver = model_driver.sort_values("lap")
pit_driver = pit_driver.sort_values("lap")

print("RAW LAP RECORDS:", len(raw_driver))
print("MODELLING ROWS:", len(model_driver))
print("LAST RECORDED LAP:", raw_driver["lap"].max())

print("\nFIRST 12 RAW LAP RECORDS")
display(raw_driver.head(12))

print("\nFIRST 10 MODELLING ROWS")
display(
    model_driver[
        ["lap", "lap_time", "lap_time_seconds",
         "reference_pace", "stint_lap", "delta"]
    ].head(10)
)

print("\nACTUAL PIT STOPS")
display(
    pit_driver[
        ["lap", "stop", "duration_raw", "duration_seconds"]
    ]
)

print("\nREFERENCE PACE:")
print(model_driver["reference_pace"].iloc[0])


RAW LAP RECORDS: 57
MODELLING ROWS: 46
LAST RECORDED LAP: 57

FIRST 12 RAW LAP RECORDS


,season,round,lap,driver_id,position,lap_time
0,2023,1,1,max_verstappen,1,1:39.019
20,2023,1,2,max_verstappen,1,1:37.974
40,2023,1,3,max_verstappen,1,1:38.006
60,2023,1,4,max_verstappen,1,1:37.976
80,2023,1,5,max_verstappen,1,1:38.035
100,2023,1,6,max_verstappen,1,1:37.986
120,2023,1,7,max_verstappen,1,1:38.021
140,2023,1,8,max_verstappen,1,1:38.154
160,2023,1,9,max_verstappen,1,1:38.278
180,2023,1,10,max_verstappen,1,1:38.369



FIRST 10 MODELLING ROWS


,lap,lap_time,lap_time_seconds,reference_pace,stint_lap,delta
388,7,1:38.021,98.021,97.9954,7,0.0256
389,8,1:38.154,98.154,97.9954,8,0.1586
390,9,1:38.278,98.278,97.9954,9,0.2826
391,10,1:38.369,98.369,97.9954,10,0.3736
392,11,1:38.483,98.483,97.9954,11,0.4876
393,12,1:38.591,98.591,97.9954,12,0.5956
394,13,1:38.482,98.482,97.9954,13,0.4866
395,16,1:37.801,97.801,97.9954,2,-0.1944
396,17,1:37.648,97.648,97.9954,3,-0.3474
397,18,1:37.614,97.614,97.9954,4,-0.3814



ACTUAL PIT STOPS


,lap,stop,duration_raw,duration_seconds
14,14,1,24.289,24.289
39,36,2,24.910,24.910



REFERENCE PACE:
97.9954


In [8]:

# ============================================================
# CELL G — VERIFY REFERENCE PHASE AND PIT-LAP EXCLUSIONS
# ============================================================

# Reconstruct the first five eligible laps from raw data
raw_driver["lap_time_seconds"] = (
    raw_driver["lap_time"]
    .str.split(":")
    .apply(lambda x: int(x[0]) * 60 + float(x[1]))
)

# Exclude lap 1 and implausible lap times
eligible_raw = raw_driver[
    (raw_driver["lap"] > 1)
    & (raw_driver["lap_time_seconds"].between(50, 200))
].copy()

reference_laps = eligible_raw.head(5)

calculated_reference = reference_laps[
    "lap_time_seconds"
].mean()

print("FIRST FIVE ELIGIBLE REFERENCE LAPS")
display(
    reference_laps[
        ["lap", "lap_time", "lap_time_seconds"]
    ]
)

print("\nCalculated reference pace:", round(calculated_reference, 4))
print(
    "Stored reference pace:",
    round(float(model_driver["reference_pace"].iloc[0]), 4)
)

print("\nACTUAL PIT LAPS AND FOLLOWING LAPS")
for pit_lap in sorted(pit_driver["lap"].tolist()):
    print(
        f"Pit lap {pit_lap}: "
        f"excluded; following lap {pit_lap + 1}: "
        "excluded from clean pace modelling"
    )

print("\nMODELLING ROWS AROUND PIT STOPS")
for pit_lap in sorted(pit_driver["lap"].tolist()):
    display(
        model_driver[
            model_driver["lap"].between(pit_lap - 2, pit_lap + 3)
        ][
            ["lap", "lap_time_seconds", "stint_lap", "delta"]
        ]
    )


FIRST FIVE ELIGIBLE REFERENCE LAPS


,lap,lap_time,lap_time_seconds
20,2,1:37.974,97.974
40,3,1:38.006,98.006
60,4,1:37.976,97.976
80,5,1:38.035,98.035
100,6,1:37.986,97.986



Calculated reference pace: 97.9954
Stored reference pace: 97.9954

ACTUAL PIT LAPS AND FOLLOWING LAPS
Pit lap 14: excluded; following lap 15: excluded from clean pace modelling
Pit lap 36: excluded; following lap 37: excluded from clean pace modelling

MODELLING ROWS AROUND PIT STOPS


,lap,lap_time_seconds,stint_lap,delta
393,12,98.591,12,0.5956
394,13,98.482,13,0.4866
395,16,97.801,2,-0.1944
396,17,97.648,3,-0.3474


,lap,lap_time_seconds,stint_lap,delta
413,34,97.839,20,-0.1564
414,35,97.850,21,-0.1454
415,38,96.297,2,-1.6984
416,39,96.446,3,-1.5494


In [9]:

# ============================================================
# CELL H — FIRST REAL SIMULATOR ENGINE
# ============================================================

def simulate_strategy(
    n_laps,
    pit_laps,
    reference_pace,
    pit_loss,
    model,
    start_lap=7
):
    """
    Estimate remaining race time under a hypothetical pit strategy.

    Reference phase: laps 1-6 already completed.
    Pit lap and immediate post-pit lap: reference-pace approximation.
    Other laps: linear Ridge prediction.
    Total pit loss: explicit user assumption.
    """

    pit_laps = sorted(list(pit_laps))

    if len(pit_laps) != len(set(pit_laps)):
        raise ValueError("Duplicate pit laps are not allowed.")

    if any(p < start_lap or p >= n_laps for p in pit_laps):
        raise ValueError("Pit laps must be within the remaining race.")

    records = []

    for lap in range(start_lap, n_laps + 1):

        # Pit lap and the following lap are excluded from pace modelling
        excluded_laps = set(pit_laps) | {
            p + 1 for p in pit_laps
        }

        if lap in excluded_laps:
            stint_lap = np.nan
            predicted_delta = 0.0
            estimated_lap_time = reference_pace

            if lap in pit_laps:
                lap_type = "Pit lap approximation"
            else:
                lap_type = "Post-pit lap approximation"

        else:
            previous_stops = [
                p for p in pit_laps if p < lap
            ]

            last_pit = max(previous_stops) if previous_stops else 0

            # Match our historical data convention:
            # lap 16 after a lap-14 stop -> stint_lap = 2
            stint_lap = lap - last_pit

            features = pd.DataFrame({
                "lap": [lap],
                "stint_lap": [stint_lap]
            })

            predicted_delta = float(
                model.predict(features)[0]
            )

            estimated_lap_time = (
                reference_pace + predicted_delta
            )

            lap_type = "Modelled lap"

        records.append({
            "lap": lap,
            "stint_lap": stint_lap,
            "lap_type": lap_type,
            "predicted_delta": predicted_delta,
            "estimated_lap_time": estimated_lap_time
        })

    lap_details = pd.DataFrame(records)

    pit_time_loss = len(pit_laps) * pit_loss

    estimated_time = (
        lap_details["estimated_lap_time"].sum()
        + pit_time_loss
    )

    summary = {
        "pit_laps": pit_laps,
        "number_of_stops": len(pit_laps),
        "pit_loss_seconds": pit_time_loss,
        "estimated_remaining_time_seconds": estimated_time
    }

    return summary, lap_details


# ------------------------------------------------------------
# Test on Verstappen's actual historical pit laps
# ------------------------------------------------------------

actual_pits = [14, 36]

summary, lap_details = simulate_strategy(
    n_laps=57,
    pit_laps=actual_pits,
    reference_pace=97.9954,
    pit_loss=22.0,
    model=linear_model,
    start_lap=7
)

print("SIMULATOR TEST — 2023 BAHRAIN GP")
print("--------------------------------")
print("Driver:", DRIVER)
print("Actual pit laps:", actual_pits)
print("Assumed total pit loss per stop: 22 seconds")

print("\nSUMMARY")
display(pd.DataFrame([summary]))

print("\nLAP DETAILS AROUND PIT STOPS")
display(
    lap_details[
        lap_details["lap"].between(12, 18)
        | lap_details["lap"].between(34, 40)
    ].round(4)
)

print("\nLAP ACCOUNTING")
print(lap_details["lap_type"].value_counts())


SIMULATOR TEST — 2023 BAHRAIN GP
--------------------------------
Driver: max_verstappen
Actual pit laps: [14, 36]
Assumed total pit loss per stop: 22 seconds

SUMMARY


,pit_laps,number_of_stops,pit_loss_seconds,estimated_remaining_time_seconds
0,"[14, 36]",2,44.0,4978.139091



LAP DETAILS AROUND PIT STOPS


,lap,stint_lap,lap_type,predicted_delta,estimated_lap_time
5,12,12.0,Modelled lap,-0.6129,97.3825
6,13,13.0,Modelled lap,-0.6311,97.3643
7,14,NaN,Pit lap approximation,0.0000,97.9954
8,15,NaN,Post-pit lap approximation,0.0000,97.9954
9,16,2.0,Modelled lap,-0.9272,97.0682
10,17,3.0,Modelled lap,-0.9454,97.0500
11,18,4.0,Modelled lap,-0.9636,97.0318
27,34,20.0,Modelled lap,-1.2545,96.7409
28,35,21.0,Modelled lap,-1.2727,96.7227
29,36,NaN,Pit lap approximation,0.0000,97.9954



LAP ACCOUNTING
lap_type
Modelled lap                  47
Pit lap approximation          2
Post-pit lap approximation     2
Name: count, dtype: int64


In [10]:

# ============================================================
# CELL I — STRATEGY COMPARISON
# ============================================================

N_LAPS = 57
REFERENCE_PACE = 97.9954
PIT_LOSS = 22.0
MIN_STINT = 8

def generate_strategies(n_laps, n_stops, min_stint=8):
    """Generate pit plans satisfying the minimum stint constraint."""

    possible_pits = range(
        min_stint,
        n_laps - min_stint + 1
    )

    for pits in itertools.combinations(possible_pits, n_stops):

        stint_lengths = np.diff([0, *pits, n_laps])

        if len(stint_lengths) and stint_lengths.min() >= min_stint:
            yield pits


def evaluate_strategy(pit_laps):
    summary, details = simulate_strategy(
        n_laps=N_LAPS,
        pit_laps=pit_laps,
        reference_pace=REFERENCE_PACE,
        pit_loss=PIT_LOSS,
        model=linear_model,
        start_lap=7
    )

    return summary, details


# ------------------------------------------------------------
# Find lowest estimated time for each stop count
# ------------------------------------------------------------

strategy_results = []

for stops in range(3):

    candidates = list(
        generate_strategies(
            N_LAPS,
            stops,
            MIN_STINT
        )
    )

    evaluated = []

    for pits in candidates:
        summary, details = evaluate_strategy(pits)

        evaluated.append({
            "pit_laps": pits,
            "estimated_time": summary[
                "estimated_remaining_time_seconds"
            ]
        })

    best = min(
        evaluated,
        key=lambda x: x["estimated_time"]
    )

    strategy_results.append({
        "strategy": f"{stops}-stop",
        "pit_laps": best["pit_laps"],
        "estimated_remaining_time_seconds": best["estimated_time"]
    })


# ------------------------------------------------------------
# Add actual historical strategy for comparison
# ------------------------------------------------------------

actual_summary, actual_details = evaluate_strategy([14, 36])

strategy_results.append({
    "strategy": "Historical actual",
    "pit_laps": (14, 36),
    "estimated_remaining_time_seconds":
        actual_summary["estimated_remaining_time_seconds"]
})


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

strategy_comparison = pd.DataFrame(strategy_results)

strategy_comparison = strategy_comparison.sort_values(
    "estimated_remaining_time_seconds"
).reset_index(drop=True)

print("2023 BAHRAIN GP — STRATEGY COMPARISON")
print("-------------------------------------")
print("Assumed pit loss:", PIT_LOSS, "seconds per stop")
print("Minimum stint:", MIN_STINT, "laps")

display(strategy_comparison.round(2))

print(
    "\nIMPORTANT: No-stop is a reference only. "
    "Times are hypothetical estimates, not actual race results."
)


2023 BAHRAIN GP — STRATEGY COMPARISON
-------------------------------------
Assumed pit loss: 22.0 seconds per stop
Minimum stint: 8 laps


,strategy,pit_laps,estimated_remaining_time_seconds
0,0-stop,(),4947.96
1,1-stop,"(27,)",4958.24
2,2-stop,"(15, 34)",4978.05
3,Historical actual,"(14, 36)",4978.14



IMPORTANT: No-stop is a reference only. Times are hypothetical estimates, not actual race results.


In [11]:

# ============================================================
# CELL J — PIT-LOSS SENSITIVITY ANALYSIS
# ============================================================

PIT_LOSS_VALUES = [0, 10, 15, 18, 22, 26, 30, 35]

sensitivity_results = []

for loss in PIT_LOSS_VALUES:

    best_by_stop_count = {}

    for stops in [1, 2]:

        candidates = list(
            generate_strategies(
                N_LAPS,
                stops,
                MIN_STINT
            )
        )

        evaluated = []

        for pits in candidates:

            summary, _ = simulate_strategy(
                n_laps=N_LAPS,
                pit_laps=pits,
                reference_pace=REFERENCE_PACE,
                pit_loss=float(loss),
                model=linear_model,
                start_lap=7
            )

            evaluated.append((
                summary["estimated_remaining_time_seconds"],
                pits
            ))

        best_time, best_pits = min(evaluated)

        best_by_stop_count[stops] = (
            best_time,
            best_pits
        )

    one_time, one_pits = best_by_stop_count[1]
    two_time, two_pits = best_by_stop_count[2]

    sensitivity_results.append({
        "pit_loss_per_stop": loss,
        "best_1_stop_lap": one_pits,
        "best_1_stop_time": one_time,
        "best_2_stop_laps": two_pits,
        "best_2_stop_time": two_time,
        "two_stop_advantage_seconds": one_time - two_time
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

print("PIT-LOSS SENSITIVITY — BAHRAIN 2023")
display(sensitivity_df.round(2))


PIT-LOSS SENSITIVITY — BAHRAIN 2023


,pit_loss_per_stop,best_1_stop_lap,best_1_stop_time,best_2_stop_laps,best_2_stop_time,two_stop_advantage_seconds
0,0,"(27,)",4936.24,"(15, 34)",4934.05,2.19
1,10,"(27,)",4946.24,"(15, 34)",4954.05,-7.81
2,15,"(27,)",4951.24,"(15, 34)",4964.05,-12.81
3,18,"(27,)",4954.24,"(15, 34)",4970.05,-15.81
4,22,"(27,)",4958.24,"(15, 34)",4978.05,-19.81
5,26,"(27,)",4962.24,"(15, 34)",4986.05,-23.81
6,30,"(27,)",4966.24,"(15, 34)",4994.05,-27.81
7,35,"(27,)",4971.24,"(15, 34)",5004.05,-32.81


In [12]:

# ============================================================
# CELL K — SAVE DAY 6 SIMULATOR OUTPUTS
# ============================================================

# Save strategy comparison
strategy_path = (
    PROCESSED / "bahrain_2023_verstappen_strategy_comparison.csv"
)

strategy_comparison.to_csv(strategy_path, index=False)

# Save sensitivity analysis
sensitivity_path = (
    PROCESSED / "bahrain_2023_verstappen_sensitivity.csv"
)

sensitivity_df.to_csv(sensitivity_path, index=False)

# Save lap-level details for the historical actual pit plan
lap_details_path = (
    PROCESSED / "bahrain_2023_verstappen_actual_plan_lap_details.csv"
)

actual_details.to_csv(lap_details_path, index=False)

print("DAY 6 OUTPUTS SAVED")
print("------------------------------")
print(strategy_path)
print(sensitivity_path)
print(lap_details_path)

print("\nSaved file sizes:")
for path in [strategy_path, sensitivity_path, lap_details_path]:
    print(f"{path.name}: {path.stat().st_size:,} bytes")


DAY 6 OUTPUTS SAVED
------------------------------
D:\projects\f1-project\data\processed\bahrain_2023_verstappen_strategy_comparison.csv
D:\projects\f1-project\data\processed\bahrain_2023_verstappen_sensitivity.csv
D:\projects\f1-project\data\processed\bahrain_2023_verstappen_actual_plan_lap_details.csv

Saved file sizes:
bahrain_2023_verstappen_strategy_comparison.csv: 199 bytes
bahrain_2023_verstappen_sensitivity.csv: 729 bytes
bahrain_2023_verstappen_actual_plan_lap_details.csv: 3,001 bytes


In [13]:

# ============================================================
# DAY 7 — CELL A: HELD-OUT RACE PREDICTIONS
# ============================================================

# Arrays for out-of-fold predictions
ridge_predictions = np.zeros(len(data))
baseline_predictions = np.zeros(len(data))

logo = LeaveOneGroupOut()

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X_linear, y, groups), start=1
):

    X_train = X_linear.iloc[train_idx]
    X_test = X_linear.iloc[test_idx]

    y_train = y[train_idx]

    # Train only on the other races
    fold_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    )

    fold_model.fit(X_train, y_train)

    # Predict the completely held-out race
    ridge_predictions[test_idx] = fold_model.predict(X_test)

    # Training-mean baseline
    baseline_predictions[test_idx] = y_train.mean()

# ------------------------------------------------------------
# Build the out-of-fold prediction dataset
# ------------------------------------------------------------

backtest = data[
    [
        "season",
        "round",
        "race_id",
        "driver_id",
        "lap",
        "stint_lap",
        "reference_pace",
        "lap_time_seconds",
        "delta"
    ]
].copy()

backtest["baseline_predicted_delta"] = baseline_predictions
backtest["ridge_predicted_delta"] = ridge_predictions

# Convert predicted delta into predicted lap time
backtest["baseline_predicted_lap_time"] = (
    backtest["reference_pace"]
    + backtest["baseline_predicted_delta"]
)

backtest["ridge_predicted_lap_time"] = (
    backtest["reference_pace"]
    + backtest["ridge_predicted_delta"]
)

# Prediction errors
backtest["baseline_error"] = (
    backtest["lap_time_seconds"]
    - backtest["baseline_predicted_lap_time"]
)

backtest["ridge_error"] = (
    backtest["lap_time_seconds"]
    - backtest["ridge_predicted_lap_time"]
)

# ------------------------------------------------------------
# Overall evaluation
# ------------------------------------------------------------

baseline_mae = mean_absolute_error(
    backtest["lap_time_seconds"],
    backtest["baseline_predicted_lap_time"]
)

ridge_mae = mean_absolute_error(
    backtest["lap_time_seconds"],
    backtest["ridge_predicted_lap_time"]
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        backtest["lap_time_seconds"],
        backtest["baseline_predicted_lap_time"]
    )
)

ridge_rmse = np.sqrt(
    mean_squared_error(
        backtest["lap_time_seconds"],
        backtest["ridge_predicted_lap_time"]
    )
)

backtest_summary = pd.DataFrame([
    {
        "Model": "Training mean baseline",
        "MAE": baseline_mae,
        "RMSE": baseline_rmse
    },
    {
        "Model": "Linear Ridge",
        "MAE": ridge_mae,
        "RMSE": ridge_rmse
    }
])

print("HELD-OUT RACE BACKTEST")
print("------------------------------")
print("Races:", backtest["race_id"].nunique())
print("Prediction rows:", len(backtest))

print("\nOVERALL PERFORMANCE")
display(backtest_summary.round(4))

improvement = (
    (baseline_mae - ridge_mae) / baseline_mae
) * 100

print(
    f"\nRidge MAE improvement over baseline: "
    f"{improvement:.2f}%"
)

# ------------------------------------------------------------
# Save out-of-fold predictions
# ------------------------------------------------------------

backtest_path = (
    PROCESSED / "heldout_race_lap_predictions_2023_2024.csv"
)

backtest.to_csv(backtest_path, index=False)

print("\nSaved:", backtest_path)


HELD-OUT RACE BACKTEST
------------------------------
Races: 35
Prediction rows: 29901

OVERALL PERFORMANCE


,Model,MAE,RMSE
0,Training mean baseline,1.0803,1.4721
1,Linear Ridge,0.9491,1.3983



Ridge MAE improvement over baseline: 12.15%

Saved: D:\projects\f1-project\data\processed\heldout_race_lap_predictions_2023_2024.csv


In [14]:

# ============================================================
# DAY 7 — CELL B: RACE-WISE PERFORMANCE
# ============================================================

race_results = (
    backtest.groupby(["season", "round", "race_id"])
    .agg(
        test_rows=("delta", "size"),
        baseline_MAE=("baseline_error", lambda x: x.abs().mean()),
        ridge_MAE=("ridge_error", lambda x: x.abs().mean()),
        baseline_RMSE=("baseline_error",
                       lambda x: np.sqrt(np.mean(x ** 2))),
        ridge_RMSE=("ridge_error",
                    lambda x: np.sqrt(np.mean(x ** 2)))
    )
    .reset_index()
)

race_results["MAE_improvement_pct"] = (
    (race_results["baseline_MAE"] - race_results["ridge_MAE"])
    / race_results["baseline_MAE"]
) * 100

race_results["ridge_beats_baseline"] = (
    race_results["ridge_MAE"] < race_results["baseline_MAE"]
)

print("RACE-WISE BACKTEST RESULTS")
display(
    race_results.sort_values("ridge_MAE", ascending=False).round(3)
)

# ------------------------------------------------------------
# Summary statistics
# ------------------------------------------------------------

wins = int(race_results["ridge_beats_baseline"].sum())
total = len(race_results)

print("\nRACE-WISE SUMMARY")
print("------------------------------")
print(f"Races where Ridge beats baseline: {wins}/{total}")
print(f"Percentage: {wins / total * 100:.2f}%")
print(
    "Median race-wise Ridge MAE:",
    round(race_results["ridge_MAE"].median(), 4)
)
print(
    "Worst race MAE:",
    round(race_results["ridge_MAE"].max(), 4)
)
print(
    "Best race MAE:",
    round(race_results["ridge_MAE"].min(), 4)
)

# ------------------------------------------------------------
# Identify strongest and weakest races
# ------------------------------------------------------------

print("\nFIVE MOST DIFFICULT RACES")
display(
    race_results.nlargest(5, "ridge_MAE")[
        ["race_id", "test_rows", "baseline_MAE",
         "ridge_MAE", "MAE_improvement_pct"]
    ].round(3)
)

print("\nFIVE BEST RACES")
display(
    race_results.nsmallest(5, "ridge_MAE")[
        ["race_id", "test_rows", "baseline_MAE",
         "ridge_MAE", "MAE_improvement_pct"]
    ].round(3)
)

# Save race-wise results
race_results.to_csv(
    PROCESSED / "heldout_race_wise_results_2023_2024.csv",
    index=False
)

print("\nRace-wise results saved.")


RACE-WISE BACKTEST RESULTS


,season,round,race_id,test_rows,baseline_MAE,ridge_MAE,baseline_RMSE,ridge_RMSE,MAE_improvement_pct,ridge_beats_baseline
14,2023,19,2023_R19,528,3.045,3.435,3.128,3.502,-12.794,False
15,2023,20,2023_R20,322,3.218,3.174,4.466,4.583,1.376,True
23,2024,8,2024_R8,789,2.523,2.997,2.750,3.153,-18.797,False
25,2024,11,2024_R11,1172,1.381,1.619,1.594,1.796,-17.269,False
33,2024,21,2024_R21,829,1.610,1.507,2.172,2.041,6.395,True
22,2024,7,2024_R7,1065,1.300,1.274,1.515,1.515,1.965,True
26,2024,12,2024_R12,546,1.727,1.247,2.323,2.010,27.804,True
10,2023,12,2023_R12,627,1.347,1.198,1.708,1.552,11.067,True
4,2023,6,2023_R6,872,1.143,1.172,1.790,1.711,-2.482,False
11,2023,14,2023_R14,790,1.126,0.991,1.313,1.186,11.913,True



RACE-WISE SUMMARY
------------------------------
Races where Ridge beats baseline: 31/35
Percentage: 88.57%
Median race-wise Ridge MAE: 0.8371
Worst race MAE: 3.4351
Best race MAE: 0.4713

FIVE MOST DIFFICULT RACES


,race_id,test_rows,baseline_MAE,ridge_MAE,MAE_improvement_pct
14,2023_R19,528,3.045,3.435,-12.794
15,2023_R20,322,3.218,3.174,1.376
23,2024_R8,789,2.523,2.997,-18.797
25,2024_R11,1172,1.381,1.619,-17.269
33,2024_R21,829,1.610,1.507,6.395



FIVE BEST RACES


,race_id,test_rows,baseline_MAE,ridge_MAE,MAE_improvement_pct
8,2023_R10,701,0.767,0.471,38.567
3,2023_R5,978,0.698,0.478,31.446
16,2023_R22,963,0.791,0.539,31.900
21,2024_R6,844,0.851,0.541,36.342
19,2024_R3,775,0.858,0.561,34.585



Race-wise results saved.


In [15]:

# ============================================================
# DAY 7 — CELL C: STINT-LEVEL BACKTEST
# ============================================================

# Group observations by stint-lap ranges
bins = [1, 5, 10, 15, 20, 25, 30, 40, 60]

labels = [
    "2-5",
    "6-10",
    "11-15",
    "16-20",
    "21-25",
    "26-30",
    "31-40",
    "41-58"
]

backtest["stint_group"] = pd.cut(
    backtest["stint_lap"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Calculate metrics for each stint group
stint_results = (
    backtest.groupby("stint_group", observed=True)
    .agg(
        observations=("delta", "size"),
        baseline_MAE=("baseline_error", lambda x: x.abs().mean()),
        ridge_MAE=("ridge_error", lambda x: x.abs().mean()),
        baseline_RMSE=("baseline_error",
                       lambda x: np.sqrt(np.mean(x ** 2))),
        ridge_RMSE=("ridge_error",
                    lambda x: np.sqrt(np.mean(x ** 2))),
        ridge_mean_error=("ridge_error", "mean")
    )
    .reset_index()
)

stint_results["MAE_improvement_pct"] = (
    (stint_results["baseline_MAE"] - stint_results["ridge_MAE"])
    / stint_results["baseline_MAE"]
) * 100

print("STINT-LEVEL MODEL PERFORMANCE")
display(stint_results.round(4))

print("\nOBSERVATION COUNTS")
display(
    stint_results[["stint_group", "observations"]]
)

# Save results
stint_results.to_csv(
    PROCESSED / "heldout_stint_level_results_2023_2024.csv",
    index=False
)

print("\nStint-level results saved.")


STINT-LEVEL MODEL PERFORMANCE


,stint_group,observations,baseline_MAE,ridge_MAE,baseline_RMSE,ridge_RMSE,ridge_mean_error,MAE_improvement_pct
0,2-5,3580,1.0940,0.9738,1.5895,1.4668,-0.2677,10.9918
1,6-10,6794,1.0423,0.7842,1.3864,1.2041,-0.0338,24.7664
2,11-15,6432,1.0810,0.8638,1.4699,1.3293,0.0539,20.0925
3,16-20,4940,1.0542,0.9308,1.4487,1.3820,0.1668,11.7061
4,21-25,3361,1.0042,0.9778,1.3529,1.3663,0.2100,2.6308
5,26-30,2130,1.0494,1.0908,1.3882,1.4488,0.1253,-3.9458
6,31-40,1996,1.2043,1.2690,1.5817,1.6513,-0.2378,-5.3706
7,41-58,668,1.6923,1.8998,2.1818,2.4234,-0.8394,-12.2604



OBSERVATION COUNTS


,stint_group,observations
0,2-5,3580
1,6-10,6794
2,11-15,6432
3,16-20,4940
4,21-25,3361
5,26-30,2130
6,31-40,1996
7,41-58,668



Stint-level results saved.


In [16]:
summary = pd.DataFrame([
    {
        "evaluation": "Overall held-out race",
        "group": "All 35 races",
        "observations": len(backtest),
        "baseline_MAE": backtest["baseline_error"].abs().mean(),
        "ridge_MAE": backtest["ridge_error"].abs().mean(),
        "baseline_RMSE": np.sqrt(np.mean(backtest["baseline_error"] ** 2)),
        "ridge_RMSE": np.sqrt(np.mean(backtest["ridge_error"] ** 2))
    },
    {
        "evaluation": "Race-wise consistency",
        "group": "35 held-out races",
        "observations": len(race_results),
        "baseline_MAE": race_results["baseline_MAE"].mean(),
        "ridge_MAE": race_results["ridge_MAE"].mean(),
        "baseline_RMSE": race_results["baseline_RMSE"].mean(),
        "ridge_RMSE": race_results["ridge_RMSE"].mean()
    }
])

summary["MAE_improvement_pct"] = (
    (summary["baseline_MAE"] - summary["ridge_MAE"])
    / summary["baseline_MAE"] * 100
)

summary.to_csv(
    PROCESSED / "backtest_summary_2023_2024.csv",
    index=False
)

print("BACKTEST SUMMARY")
display(summary.round(4))

print("\nRace-wise wins:", (race_results["ridge_beats_baseline"]).sum()
      if "ridge_beats_baseline" in race_results.columns
      else (race_results["MAE_improvement_pct"] > 0).sum())

print("\nSaved:", PROCESSED / "backtest_summary_2023_2024.csv")

BACKTEST SUMMARY


,evaluation,group,observations,baseline_MAE,ridge_MAE,baseline_RMSE,ridge_RMSE,MAE_improvement_pct
0,Overall held-out race,All 35 races,29901,1.0803,0.9491,1.4721,1.3983,12.1469
1,Race-wise consistency,35 held-out races,35,1.1578,1.0294,1.4617,1.3320,11.0886



Race-wise wins: 31

Saved: D:\projects\f1-project\data\processed\backtest_summary_2023_2024.csv


In [18]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\projects\f1-project")

print("Project folder exists:", PROJECT_ROOT.exists())
print("DECISIONS.md exists:", (PROJECT_ROOT / "DECISIONS.md").exists())
print("Current notebook directory:", Path.cwd())

Project folder exists: True
DECISIONS.md exists: True
Current notebook directory: d:\projects\f1-project\notebooks


In [20]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\projects\f1-project")
decisions_path = PROJECT_ROOT / "DECISIONS.md"

existing = decisions_path.read_text(encoding="utf-8")

day6 = """
## Day 6 — Pit-stop scenario simulator

- Built a hypothetical remaining-race simulator using the linear Ridge clean-pace model.
- Selected linear Ridge for transparency; the quadratic model's MAE advantage was only about 0.0086 seconds.
- Ordinary laps use race lap and stint-lap features. A pit lap and the following lap are approximated at reference pace.
- Total pit loss is an explicit assumption. API pit-stop duration is stationary stop duration, not total pit-lane loss.
- Demonstrated on 2023 Bahrain GP, Max Verstappen, using an assumed 22-second pit loss.
- Compared no-stop, one-stop, two-stop and historical-stop scenarios; tested sensitivity to pit loss.
- No-stop is a reference only, not a legal dry-race strategy under the two-compound rule.
- The simulator does not model compounds, physical tyre degradation, fuel, traffic, weather or safety cars.
- The Bahrain demonstration model was trained on all 35 eligible races, including Bahrain. This is an in-sample demonstration.

"""

day7 = """
## Day 7 — Historical backtesting

- Generated out-of-fold predictions by holding out one complete race at a time and training on the other 34 races.
- Evaluated 35 eligible races and 29,901 clean prediction rows.
- Training-mean baseline: MAE 1.0803 seconds; RMSE 1.4721 seconds.
- Linear Ridge: MAE 0.9491 seconds; RMSE 1.3983 seconds.
- Ridge reduced pooled MAE by 12.15% and beat the baseline in 31 of 35 races (88.6%).
- Mean race-level MAE: 1.1578 seconds for baseline and 1.0294 seconds for Ridge.
- Ridge performed better in stint groups 2–25; the baseline performed better in groups 26–58.
- Long-stint results are a limitation. The model lacks tyre compound, verified tyre age, fuel, traffic, weather and safety-car information.
- Reference pace is derived retrospectively from each driver-race's first five eligible laps. This is not a live race-start forecast.
- These results validate clean-pace prediction on the selected historical dataset; they do not yet validate the strategy simulator's race-time accuracy.

"""

if "## Day 6 — Pit-stop scenario simulator" not in existing:
    with decisions_path.open("a", encoding="utf-8") as f:
        f.write("\n" + day6)

if "## Day 7 — Historical backtesting" not in existing:
    with decisions_path.open("a", encoding="utf-8") as f:
        f.write("\n" + day7)

updated = decisions_path.read_text(encoding="utf-8")

print("DECISIONS.md update checked.")
print("Day 6 section present:", "## Day 6 — Pit-stop scenario simulator" in updated)
print("Day 7 section present:", "## Day 7 — Historical backtesting" in updated)
print("File:", decisions_path)

DECISIONS.md update checked.
Day 6 section present: True
Day 7 section present: True
File: D:\projects\f1-project\DECISIONS.md
